# STAMP Multi-Floor Shortest-Path Tool

Combines the **basement**, **ground**, and **first** floors of the STAMP
building (UMD) into a single `networkx` graph and finds the shortest path
between any two nodes — possibly across floors — with optional bathroom
stops and accessibility (no-stairs) routing.

Workflow:
1. Pick a starting **floor**, then a starting **node** — see the highlighted base map.
2. Pick an ending **floor** and **node** — see the destination floor.
3. Toggle **bathroom stop** and **accessible path** as needed.
4. Click **Find Shortest Path** to see the route on each floor it touches.


In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output


## 1. Node table

Each node has `floor`, `type`, and `pos` (Excel coords ÷ 100 so the
floorplan plots use sensible decimal numbers). Node IDs are unique
across floors (1–45 ground, 46–94 first, 95–130 basement) — taken
directly from the source spreadsheet.

In [2]:
# (node_id) -> {'floor', 'type', 'pos'}
NODES = {
    # ─── GROUND FLOOR (G) ─────────────────────────────────────────
    1:  {'floor': 'ground',   'type': 'stairs',   'pos': (12.75,  1.50)},  # SE staircase
    2:  {'floor': 'ground',   'type': 'hallway',  'pos': (12.75,  1.75)},
    3:  {'floor': 'ground',   'type': 'hallway',  'pos': (11.50,  1.75)},
    4:  {'floor': 'ground',   'type': 'hallway',  'pos': (11.25,  1.75)},
    5:  {'floor': 'ground',   'type': 'hallway',  'pos': (11.00,  0.50)},
    6:  {'floor': 'ground',   'type': 'elevator', 'pos': (10.90,  0.60)},  # S elevator
    7:  {'floor': 'ground',   'type': 'stairs',   'pos': (10.50,  0.50)},  # S staircase
    8:  {'floor': 'ground',   'type': 'hallway',  'pos': ( 9.00,  1.75)},
    9:  {'floor': 'ground',   'type': 'hallway',  'pos': ( 9.00,  1.50)},
    10: {'floor': 'ground',   'type': 'elevator', 'pos': ( 8.75,  1.60)},
    11: {'floor': 'ground',   'type': 'hallway',  'pos': ( 7.75,  1.75)},
    12: {'floor': 'ground',   'type': 'stairs',   'pos': ( 7.50,  2.50)},
    13: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.50,  1.75)},
    14: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.50,  2.50)},
    15: {'floor': 'ground',   'type': 'elevator', 'pos': ( 3.50,  2.50)},
    16: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.50,  2.75)},
    17: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.15,  2.80)},
    18: {'floor': 'ground',   'type': 'stairs',   'pos': ( 3.50,  2.80)},
    19: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.15,  3.50)},
    20: {'floor': 'ground',   'type': 'stairs',   'pos': ( 3.50,  3.50)},
    21: {'floor': 'ground',   'type': 'stairs',   'pos': (11.25,  2.75)},
    22: {'floor': 'ground',   'type': 'hallway',  'pos': ( 7.75,  7.50)},
    23: {'floor': 'ground',   'type': 'hallway',  'pos': ( 8.00, 10.25)},
    24: {'floor': 'ground',   'type': 'stairs',   'pos': ( 8.75, 10.25)},
    25: {'floor': 'ground',   'type': 'elevator', 'pos': ( 8.00, 10.75)},
    26: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.00,  7.50)},
    27: {'floor': 'ground',   'type': 'entrance', 'pos': ( 3.25,  7.50)},
    28: {'floor': 'ground',   'type': 'hallway',  'pos': ( 4.00,  6.50)},
    29: {'floor': 'ground',   'type': 'stairs',   'pos': ( 3.50,  6.60)},
    30: {'floor': 'ground',   'type': 'hallway',  'pos': ( 9.50,  7.75)},
    31: {'floor': 'ground',   'type': 'stairs',   'pos': ( 9.50,  7.50)},
    32: {'floor': 'ground',   'type': 'stairs',   'pos': (14.75,  8.00)},
    33: {'floor': 'ground',   'type': 'elevator', 'pos': (15.00,  7.75)},
    34: {'floor': 'ground',   'type': 'hallway',  'pos': (14.75,  4.75)},
    35: {'floor': 'ground',   'type': 'entrance', 'pos': (15.25,  4.75)},
    36: {'floor': 'ground',   'type': 'hallway',  'pos': (14.75,  1.75)},
    37: {'floor': 'ground',   'type': 'entrance', 'pos': (14.75,  1.50)},
    38: {'floor': 'ground',   'type': 'bathroom', 'pos': ( 5.75,  7.75)},  # W bathroom
    39: {'floor': 'ground',   'type': 'bathroom', 'pos': (12.25,  8.00)},  # NE bathroom 1
    40: {'floor': 'ground',   'type': 'bathroom', 'pos': (12.90,  8.00)},  # NE bathroom 2
    41: {'floor': 'ground',   'type': 'bathroom', 'pos': ( 5.25,  2.25)},  # SW bathroom
    42: {'floor': 'ground',   'type': 'hallway',  'pos': ( 5.75,  7.30)},
    43: {'floor': 'ground',   'type': 'hallway',  'pos': (12.25,  7.75)},
    44: {'floor': 'ground',   'type': 'hallway',  'pos': (12.90,  7.75)},
    45: {'floor': 'ground',   'type': 'hallway',  'pos': ( 5.30,  1.75)},

    # ─── FIRST FLOOR (1) ──────────────────────────────────────────
    46: {'floor': 'first',    'type': 'stairs',   'pos': ( 6.25,  1.50)},
    47: {'floor': 'first',    'type': 'stairs',   'pos': ( 6.40,  1.60)},
    48: {'floor': 'first',    'type': 'entrance', 'pos': ( 6.10,  1.25)},
    49: {'floor': 'first',    'type': 'entrance', 'pos': ( 6.45,  1.25)},
    50: {'floor': 'first',    'type': 'hallway',  'pos': ( 6.45,  2.00)},
    51: {'floor': 'first',    'type': 'hallway',  'pos': ( 6.10,  2.00)},
    52: {'floor': 'first',    'type': 'hallway',  'pos': ( 5.30,  2.00)},
    53: {'floor': 'first',    'type': 'bathroom', 'pos': ( 5.30,  1.90)},
    54: {'floor': 'first',    'type': 'hallway',  'pos': ( 5.15,  2.00)},
    55: {'floor': 'first',    'type': 'hallway',  'pos': ( 5.15,  2.00)},  # same coord as n54 in source
    56: {'floor': 'first',    'type': 'stairs',   'pos': ( 5.50,  2.30)},
    57: {'floor': 'first',    'type': 'hallway',  'pos': ( 4.50,  2.00)},
    58: {'floor': 'first',    'type': 'stairs',   'pos': ( 4.50,  1.55)},
    59: {'floor': 'first',    'type': 'elevator', 'pos': ( 4.35,  1.75)},
    60: {'floor': 'first',    'type': 'entrance', 'pos': ( 3.75,  1.35)},
    61: {'floor': 'first',    'type': 'entrance', 'pos': ( 3.40,  1.35)},
    62: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.75,  2.00)},
    63: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.40,  2.00)},
    64: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.20,  2.00)},
    65: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.70,  2.00)},
    66: {'floor': 'first',    'type': 'stairs',   'pos': ( 2.70,  1.55)},
    67: {'floor': 'first',    'type': 'elevator', 'pos': ( 2.60,  1.60)},
    68: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.40,  2.00)},
    69: {'floor': 'first',    'type': 'bathroom', 'pos': ( 2.40,  1.80)},
    70: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.05,  2.00)},
    71: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.05,  2.30)},
    72: {'floor': 'first',    'type': 'elevator', 'pos': ( 1.85,  2.30)},
    73: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.05,  2.50)},
    74: {'floor': 'first',    'type': 'stairs',   'pos': ( 1.80,  2.50)},
    75: {'floor': 'first',    'type': 'hallway',  'pos': ( 2.05,  2.80)},
    76: {'floor': 'first',    'type': 'stairs',   'pos': ( 1.80,  2.80)},
    77: {'floor': 'first',    'type': 'entrance', 'pos': ( 1.60,  2.50)},
    78: {'floor': 'first',    'type': 'entrance', 'pos': ( 1.60,  2.80)},
    79: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.25,  2.30)},
    80: {'floor': 'first',    'type': 'stairs',   'pos': ( 3.60,  2.30)},  # main staircase
    81: {'floor': 'first',    'type': 'hallway',  'pos': ( 4.00,  2.30)},
    82: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.25,  4.35)},
    83: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.90,  4.35)},
    84: {'floor': 'first',    'type': 'stairs',   'pos': ( 1.75,  4.40)},
    85: {'floor': 'first',    'type': 'hallway',  'pos': ( 3.25,  4.85)},
    86: {'floor': 'first',    'type': 'hallway',  'pos': ( 4.00,  4.85)},
    87: {'floor': 'first',    'type': 'hallway',  'pos': ( 4.00,  6.00)},
    88: {'floor': 'first',    'type': 'elevator', 'pos': ( 4.00,  6.25)},
    89: {'floor': 'first',    'type': 'stairs',   'pos': ( 4.25,  6.10)},
    90: {'floor': 'first',    'type': 'hallway',  'pos': ( 4.75,  4.85)},
    91: {'floor': 'first',    'type': 'stairs',   'pos': ( 4.75,  4.50)},
    92: {'floor': 'first',    'type': 'bathroom', 'pos': ( 4.90,  5.00)},
    93: {'floor': 'first',    'type': 'stairs',   'pos': ( 7.20,  4.90)},
    94: {'floor': 'first',    'type': 'elevator', 'pos': ( 7.30,  4.85)},

    # ─── BASEMENT (B) ─────────────────────────────────────────────
    95:  {'floor': 'basement', 'type': 'entrance', 'pos': ( 9.10,  4.00)},
    96:  {'floor': 'basement', 'type': 'elevator', 'pos': ( 9.00,  4.50)},
    97:  {'floor': 'basement', 'type': 'hallway',  'pos': ( 8.70,  4.40)},
    98:  {'floor': 'basement', 'type': 'stairs',   'pos': ( 8.60,  4.60)},
    99:  {'floor': 'basement', 'type': 'hallway',  'pos': ( 8.50,  4.50)},
    100: {'floor': 'basement', 'type': 'hallway',  'pos': ( 7.10,  4.50)},
    101: {'floor': 'basement', 'type': 'hallway',  'pos': ( 6.00,  4.50)},
    102: {'floor': 'basement', 'type': 'stairs',   'pos': ( 6.00,  3.80)},
    103: {'floor': 'basement', 'type': 'hallway',  'pos': ( 4.75,  4.50)},
    104: {'floor': 'basement', 'type': 'hallway',  'pos': ( 4.35,  5.15)},
    105: {'floor': 'basement', 'type': 'hallway',  'pos': ( 4.75,  5.95)},
    106: {'floor': 'basement', 'type': 'elevator', 'pos': ( 4.75,  6.15)},
    107: {'floor': 'basement', 'type': 'hallway',  'pos': ( 4.30,  6.00)},
    108: {'floor': 'basement', 'type': 'stairs',   'pos': ( 4.05,  6.00)},
    109: {'floor': 'basement', 'type': 'stairs',   'pos': ( 5.25,  6.00)},
    110: {'floor': 'basement', 'type': 'entrance', 'pos': ( 5.00,  6.25)},
    111: {'floor': 'basement', 'type': 'entrance', 'pos': ( 4.30,  6.25)},
    112: {'floor': 'basement', 'type': 'stairs',   'pos': ( 4.00,  5.50)},
    113: {'floor': 'basement', 'type': 'bathroom', 'pos': ( 4.35,  4.45)},
    114: {'floor': 'basement', 'type': 'hallway',  'pos': ( 3.80,  4.50)},
    115: {'floor': 'basement', 'type': 'hallway',  'pos': ( 3.80,  3.80)},
    116: {'floor': 'basement', 'type': 'hallway',  'pos': ( 2.35,  3.60)},
    117: {'floor': 'basement', 'type': 'hallway',  'pos': ( 3.80,  3.40)},
    118: {'floor': 'basement', 'type': 'bathroom', 'pos': ( 4.25,  3.40)},
    119: {'floor': 'basement', 'type': 'hallway',  'pos': ( 3.25,  3.40)},
    120: {'floor': 'basement', 'type': 'stairs',   'pos': ( 1.85,  3.75)},
    121: {'floor': 'basement', 'type': 'hallway',  'pos': ( 2.40,  2.70)},
    122: {'floor': 'basement', 'type': 'hallway',  'pos': ( 2.40,  1.90)},
    123: {'floor': 'basement', 'type': 'stairs',   'pos': ( 2.00,  1.90)},
    124: {'floor': 'basement', 'type': 'hallway',  'pos': ( 5.85,  2.60)},
    125: {'floor': 'basement', 'type': 'hallway',  'pos': ( 7.15,  1.75)},
    126: {'floor': 'basement', 'type': 'stairs',   'pos': ( 6.70,  1.75)},
    127: {'floor': 'basement', 'type': 'stairs',   'pos': ( 6.30,  0.25)},
    128: {'floor': 'basement', 'type': 'elevator', 'pos': ( 6.50,  0.25)},
    129: {'floor': 'basement', 'type': 'ramp',     'pos': ( 4.70,  4.00)},
    130: {'floor': 'basement', 'type': 'hallway',  'pos': ( 4.60,  3.40)},
}

FLOOR_ORDER = ['basement', 'ground', 'first']  # bottom -> top


## 2. Within-floor edges

Format: `(node_a, node_b, distance_ft, edge_type)`.
`edge_type` drives the cost penalty: `stairs`/`elevator` cost more than
`hallway`/`entrance`.

In [3]:
EDGES = [
    # ─── GROUND FLOOR EDGES (E1–E45) ──────────────────────────────
    ( 1,  2,  10, 'hallway'),    # E1
    ( 2,  3,  30, 'hallway'),    # E2
    ( 3,  4,  15, 'hallway'),    # E3
    ( 3,  5,  40, 'hallway'),    # E4
    ( 5,  6,   5, 'elevator'),   # E5
    ( 5,  7,  15, 'stairs'),     # E6
    ( 4, 21,  25, 'stairs'),     # E7
    ( 4,  8,  60, 'hallway'),    # E8
    ( 8,  9,  10, 'hallway'),    # E9
    ( 9, 10,  10, 'elevator'),   # E10
    ( 8, 11,  35, 'hallway'),    # E11
    (11, 12,  20, 'stairs'),     # E12
    (11, 45,  65, 'hallway'),    # E13
    (13, 14,  15, 'hallway'),    # E14
    (14, 15,  25, 'elevator'),   # E15
    (14, 16,  10, 'hallway'),    # E16
    (16, 17,  10, 'hallway'),    # E17
    (17, 18,  10, 'stairs'),     # E18
    (17, 19,  10, 'hallway'),    # E19
    (19, 20,  10, 'stairs'),     # E20
    (11, 22, 150, 'hallway'),    # E21
    (22, 23,  75, 'hallway'),    # E22
    (23, 24,  10, 'stairs'),     # E23
    (23, 25,   5, 'elevator'),   # E24
    (22, 26, 100, 'hallway'),    # E25
    (26, 27,  15, 'entrance'),   # E26
    (26, 28,  15, 'hallway'),    # E27
    (28, 29,  10, 'stairs'),     # E28
    (22, 30,  40, 'hallway'),    # E29
    (30, 31,  15, 'stairs'),     # E30
    (30, 32, 130, 'stairs'),     # E31
    (32, 33,   5, 'stairs'),     # E32
    (34, 33,  75, 'elevator'),   # E33
    (34, 35,   5, 'entrance'),   # E34
    (34, 36,  75, 'hallway'),    # E35
    (36, 37,   5, 'entrance'),   # E36
    (36,  2,  50, 'hallway'),    # E37
    (38, 42,   5, 'hallway'),    # E38
    (39, 43,   5, 'hallway'),    # E39
    (40, 44,   5, 'hallway'),    # E40
    (41, 45,   5, 'hallway'),    # E41
    (45, 13,  15, 'hallway'),    # E42
    (26, 42,  50, 'hallway'),    # E43
    (43, 44,  10, 'hallway'),    # E44
    (32, 44,  40, 'hallway'),    # E45

    # ─── FIRST FLOOR EDGES (E46–E96, E142) ────────────────────────
    (47, 50,  20, 'stairs'),     # E46
    (47, 49,  15, 'entrance'),   # E47
    (48, 46,  15, 'entrance'),   # E48
    (46, 51,  20, 'stairs'),     # E49
    (51, 50,  20, 'hallway'),    # E50
    (52, 51,  45, 'hallway'),    # E51
    (52, 53,   5, 'hallway'),    # E52
    (52, 54,  10, 'hallway'),    # E53
    (54, 55,  15, 'hallway'),    # E54
    (55, 56,  20, 'stairs'),     # E55
    (54, 57,  35, 'hallway'),    # E56
    (57, 58,  25, 'stairs'),     # E57
    (57, 59,  15, 'elevator'),   # E58
    (57, 62,  40, 'hallway'),    # E59
    (62, 60,  35, 'entrance'),   # E60
    (62, 63,  20, 'hallway'),    # E61
    (63, 61,  35, 'entrance'),   # E62
    (63, 64,  10, 'hallway'),    # E63
    (64, 65,  30, 'hallway'),    # E64
    (65, 66,  25, 'stairs'),     # E65
    (65, 67,  15, 'elevator'),   # E66
    (65, 68,  15, 'hallway'),    # E67
    (68, 70,  20, 'hallway'),    # E68
    (70, 71,  15, 'hallway'),    # E69
    (71, 72,  10, 'elevator'),   # E70
    (71, 73,  10, 'hallway'),    # E71
    (73, 74,  20, 'stairs'),     # E72
    (73, 75,  20, 'hallway'),    # E73
    (75, 76,  20, 'stairs'),     # E74
    (74, 77,  10, 'entrance'),   # E75
    (76, 78,  10, 'entrance'),   # E76
    (64, 79,  20, 'hallway'),    # E77
    (63, 80,  20, 'stairs'),     # E78
    (62, 80,  20, 'stairs'),     # E79
    (62, 81,  20, 'hallway'),    # E80
    (79, 80,  20, 'stairs'),     # E81
    (80, 81,  20, 'stairs'),     # E82
    (81, 83, 110, 'hallway'),    # E83
    (79, 82, 110, 'hallway'),    # E84
    (82, 84,  80, 'stairs'),     # E85
    (82, 85,  30, 'hallway'),    # E86
    (83, 86,  30, 'hallway'),    # E87
    (85, 86,  40, 'hallway'),    # E88
    (86, 87,  60, 'hallway'),    # E89
    (87, 88,  15, 'elevator'),   # E90
    (87, 89,  10, 'stairs'),     # E91
    (86, 90,  40, 'hallway'),    # E92
    (90, 91,  20, 'stairs'),     # E93
    (90, 92,  10, 'hallway'),    # E94
    (90, 93, 135, 'stairs'),     # E95
    (90, 94, 145, 'elevator'),   # E96
    (68, 69,   5, 'hallway'),    # E142

    # ─── BASEMENT EDGES (E97–E141) ────────────────────────────────
    ( 95,  97,  15, 'entrance'),  # E97
    ( 96,  97,   5, 'elevator'),  # E98
    ( 97,  99,  10, 'hallway'),   # E99
    ( 97,  98,  10, 'stairs'),    # E100
    ( 98,  99,   5, 'stairs'),    # E101
    ( 99, 100,  70, 'hallway'),   # E102
    (100, 125, 125, 'hallway'),   # E103
    (125, 126,  15, 'stairs'),    # E104
    (100, 101,  50, 'hallway'),   # E105
    (101, 102,  30, 'stairs'),    # E106
    (101, 103,  50, 'hallway'),   # E107
    (103, 105,  60, 'hallway'),   # E108
    (105, 106,  10, 'elevator'),  # E109
    (105, 110,  15, 'entrance'),  # E110
    (109, 105,  15, 'stairs'),    # E111
    (105, 107,  20, 'hallway'),   # E112
    (107, 111,   5, 'entrance'),  # E113
    (107, 108,  10, 'stairs'),    # E114
    (108, 112,  20, 'stairs'),    # E115
    (112, 104,  25, 'stairs'),    # E116
    (104, 105,  40, 'hallway'),   # E117
    (104, 103,  35, 'hallway'),   # E118
    (113, 103,  20, 'hallway'),   # E119
    (104, 113,  30, 'hallway'),   # E120
    (114, 104,  35, 'hallway'),   # E121
    (114, 113,  20, 'hallway'),   # E122
    (112, 114,  45, 'stairs'),    # E123
    (114, 115,  30, 'hallway'),   # E124
    (115, 117,  20, 'hallway'),   # E125
    (117, 118,  15, 'hallway'),   # E126
    (119, 115,  35, 'hallway'),   # E127
    (119, 117,  30, 'hallway'),   # E128
    (116, 115,  70, 'hallway'),   # E129
    (120, 116,  20, 'stairs'),    # E130
    (116, 119,  40, 'hallway'),   # E131
    (119, 121,  45, 'hallway'),   # E132
    (121, 116,  40, 'hallway'),   # E133
    (121, 122,  40, 'hallway'),   # E134
    (123, 122,  15, 'stairs'),    # E135
    (103, 124, 100, 'hallway'),   # E136
    (103, 129,  30, 'hallway'),   # E137
    (129, 130,  25, 'hallway'),   # E138
    (130, 124,  65, 'hallway'),   # E139
    (124, 127, 110, 'stairs'),    # E140
    (124, 128, 110, 'elevator'),  # E141
]


## 3. Inter-floor connections

These edges represent **shared elevator / staircase shafts** that link
two floors. Each one becomes a high-cost edge in the graph: traversing
it means physically going up or down a flight of stairs / one elevator
ride.

Mappings below are **best-guess pairings** based on labels in the source
spreadsheet (`SE staircase`, `S elevator`, `main staircase`, …) and
relative position within each floor — the ground/first/basement plans
were digitized with independent coordinate origins, so they cannot be
auto-matched. Edit the lists if you learn that a particular shaft maps
differently in the building.

In [4]:
# (lower_floor_node, upper_floor_node, transition_type)
INTER_FLOOR_EDGES = [
    # ─── BASEMENT ↔ GROUND ──────────────────────────────────
    ( 96,  33, 'elevator'),    # central-S elevator shaft
    (106,  25, 'elevator'),    # west elevator shaft
    (128,   6, 'elevator'),    # S elevator shaft
    ( 98,   32, 'stairs'),      # S staircase
    (102,  31, 'stairs'),      # central staircase
    (120, 29, 'stairs'),
    (123,  20, 'stairs'),      # NW staircase
    (126,  21, 'stairs'),      # SE staircase
    (127,  7, 'stairs'),      # central staircase

    # ─── GROUND ↔ FIRST ─────────────────────────────────────
    (  10,  59, 'elevator'),    # S elevator shaft
    ( 15,  72, 'elevator'),    # west elevator
    ( 25,  88, 'elevator'),    # N elevator
    ( 33,  94, 'elevator'),    # E elevator
    (  1,  47, 'stairs'),      # SE staircase
    (  7,  46, 'stairs'),      # S staircase
    ( 12,  56, 'stairs'),      # central staircase
    ( 18,  66, 'stairs'),      # W staircase
    ( 20,  74, 'stairs'),      # W staircase
    ( 21,  58, 'stairs'),      # central staircase
    ( 24,  89, 'stairs'),      # N staircase
    ( 29,  84, 'stairs'),      # NW staircase
    ( 31,  91, 'stairs'),      # central N staircase
    ( 32,  93, 'stairs'),      # E staircase
]

# Conceptual cost of going up/down one floor
TRANSITION_DISTANCE_STAIRS   = 30   # ft equivalent
TRANSITION_DISTANCE_ELEVATOR = 25   # ft equivalent (incl. wait)


## 4. Build the combined graph

In [5]:
G = nx.Graph()

for nid, info in NODES.items():
    G.add_node(nid, **info)

for u, v, dist, etype in EDGES:
    G.add_edge(u, v, distance=dist, edge_type=etype, congestion=1.0)

for u, v, etype in INTER_FLOOR_EDGES:
    dist = (TRANSITION_DISTANCE_STAIRS if etype == 'stairs'
            else TRANSITION_DISTANCE_ELEVATOR)
    G.add_edge(u, v, distance=dist,
               edge_type=etype + '_transition', congestion=1.0)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
for fl in FLOOR_ORDER:
    cnt = sum(1 for n, a in G.nodes(data=True) if a['floor'] == fl)
    print(f'  {fl:9s}: {cnt} nodes')


Graph: 130 nodes, 167 edges
  basement : 36 nodes
  ground   : 45 nodes
  first    : 49 nodes


## 5. Cost function and pathfinding

`make_weight_fn(accessibility=True)` returns a NetworkX-compatible
edge-weight callable that **returns `None` (= edge unusable)** for
stairs or stair-transitions when accessibility is required, so Dijkstra
will route exclusively through hallways / elevators / ramps.

In [6]:
# Edge-type penalty multipliers
PENALTY = {
    'hallway':              1.0,
    'entrance':             1.0,
    'ramp':                 1.0,
    'stairs':               1.5,
    'elevator':             2.0,
    'stairs_transition':    2.5,   # vertical stair flight
    'elevator_transition':  3.0,   # vertical elevator ride (wait time)
}


def make_weight_fn(accessibility=False):
    """Return a (u, v, data) -> cost function. None means edge is forbidden."""
    def w(u, v, data):
        et = data.get('edge_type', 'hallway')
        if accessibility and et in ('stairs', 'stairs_transition'):
            return None
        d = data.get('distance', 1)
        c = data.get('congestion', 1)
        return d * c * PENALTY.get(et, 1.0)
    return w


def shortest_path_with_options(graph, source, target,
                                accessibility=False, bathroom=False):
    """Returns (path, total_cost, bathroom_node_or_None)."""
    w = make_weight_fn(accessibility)

    if not bathroom:
        path = nx.shortest_path(graph, source, target, weight=w)
        cost = nx.shortest_path_length(graph, source, target, weight=w)
        return path, cost, None

    # bathroom stop: try every bathroom node, keep the cheapest start->bath->end
    baths = [n for n, a in graph.nodes(data=True) if a.get('type') == 'bathroom']
    best = None
    for b in baths:
        try:
            p1   = nx.shortest_path(graph, source, b, weight=w)
            c1   = nx.shortest_path_length(graph, source, b, weight=w)
            p2   = nx.shortest_path(graph, b, target, weight=w)
            c2   = nx.shortest_path_length(graph, b, target, weight=w)
        except nx.NetworkXNoPath:
            continue
        total = c1 + c2
        if best is None or total < best[1]:
            best = (p1 + p2[1:], total, b)
    if best is None:
        raise nx.NetworkXNoPath(
            f'No bathroom-stop path between {source} and {target}.')
    return best


## 6. Drawing helpers

Each floor draws independently (its own coordinate system). The path
visualizer creates one subplot per floor that the path touches.

In [7]:
POI_COLORS = {
    'elevator': '#9b59b6',
    'stairs':   '#f4d03f',
    'entrance': '#1abc9c',
    'bathroom': '#e91e63',
    'ramp':     '#ff8c00',
    'hallway':  '#bdc3c7',
}

POI_SHAPES = {
    'elevator': ('s', 600),
    'stairs':   ('^', 600),
    'entrance': ('D', 600),
    'bathroom': ('p', 600),
    'ramp':     ('h', 600),
    'hallway':  ('o', 350),
}

LEGEND_HANDLES = [
    Line2D([0],[0], marker=POI_SHAPES[t][0], color='w',
           markerfacecolor=POI_COLORS[t], markeredgecolor='#555',
           markersize=10, label=t.capitalize())
    for t in ['elevator', 'stairs', 'entrance', 'bathroom', 'ramp', 'hallway']
]


def floor_subgraph(floor):
    nodes = [n for n, a in G.nodes(data=True) if a.get('floor') == floor]
    return G.subgraph(nodes)


def _floor_pos(sub):
    return {n: G.nodes[n]['pos'] for n in sub.nodes()}


def _draw_typed_nodes(sub, pos, ax, color_fn):
    by_type = {}
    for n in sub.nodes():
        t = G.nodes[n].get('type', 'hallway')
        by_type.setdefault(t, []).append(n)
    for t, nodes in by_type.items():
        shape, size = POI_SHAPES.get(t, ('o', 350))
        nx.draw_networkx_nodes(
            sub, pos, nodelist=nodes,
            node_color=[color_fn(n) for n in nodes],
            node_shape=shape, node_size=size,
            edgecolors='#555', linewidths=1.0, ax=ax,
        )


def draw_floor_map(floor, ax=None, highlight=None, title_suffix=''):
    """Base map of a single floor with all POIs colored by type."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 7))
    sub = floor_subgraph(floor)
    pos = _floor_pos(sub)

    nx.draw_networkx_edges(sub, pos, edge_color='#cccccc', width=1.5, ax=ax)
    def color_fn(n):
        if n == highlight:
            return '#e74c3c'
        return POI_COLORS.get(G.nodes[n].get('type', 'hallway'), '#bdc3c7')
    _draw_typed_nodes(sub, pos, ax, color_fn)
    nx.draw_networkx_labels(sub, pos, font_size=8, font_weight='bold', ax=ax)

    ax.legend(handles=LEGEND_HANDLES, loc='upper right', fontsize=8,
              title='Node type', framealpha=0.85)
    ax.set_title(f'{floor.capitalize()} floor{title_suffix}')
    ax.set_aspect('equal')
    ax.axis('off')


def draw_path_on_floor(floor, path, ax,
                        source=None, target=None, bathroom_node=None):
    """Highlight the portion of `path` that lies on `floor`."""
    sub = floor_subgraph(floor)
    pos = _floor_pos(sub)

    path_set = set(path)
    path_edges = list(zip(path, path[1:]))
    on_floor_edges = [(u, v) for u, v in path_edges if u in pos and v in pos]
    on_floor_set   = set(map(frozenset, on_floor_edges))

    other_edges = [e for e in sub.edges() if frozenset(e) not in on_floor_set]
    nx.draw_networkx_edges(sub, pos, edgelist=other_edges,
                           edge_color='#eaeaea', width=1, ax=ax)
    nx.draw_networkx_edges(sub, pos, edgelist=on_floor_edges,
                           edge_color='crimson', width=4, ax=ax)

    def color_fn(n):
        if   n == source:        return '#2ecc71'
        elif n == target:        return '#3498db'
        elif n == bathroom_node: return '#ff69b4'
        elif n in path_set:      return '#f39c12'
        return POI_COLORS.get(G.nodes[n].get('type', 'hallway'), '#bdc3c7')
    _draw_typed_nodes(sub, pos, ax, color_fn)

    bold   = {n: n for n in sub.nodes() if n in path_set}
    faded  = {n: n for n in sub.nodes() if n not in path_set}
    nx.draw_networkx_labels(sub, pos, labels=faded,
                            font_size=7, font_color='#aaaaaa', ax=ax)
    nx.draw_networkx_labels(sub, pos, labels=bold,
                            font_size=9, font_weight='bold', ax=ax)

    if on_floor_edges:
        elabels = {(u, v): f"{G[u][v]['distance']}ft" for u, v in on_floor_edges}
        nx.draw_networkx_edge_labels(sub, pos, edge_labels=elabels,
                                     font_size=7, ax=ax)

    ax.set_title(f'{floor.capitalize()} floor')
    ax.set_aspect('equal')
    ax.axis('off')


def visualize_path(path, source, target, total_cost, bathroom_node=None):
    """One subplot per floor the path traverses, plus a text breakdown."""
    floors_in_path = []
    for n in path:
        f = G.nodes[n].get('floor')
        if f not in floors_in_path:
            floors_in_path.append(f)
    # display in canonical bottom->top order
    floors_in_path = [f for f in FLOOR_ORDER if f in floors_in_path]

    nf = len(floors_in_path)
    fig, axes = plt.subplots(1, nf, figsize=(7 * nf, 7))
    if nf == 1:
        axes = [axes]

    for ax, fl in zip(axes, floors_in_path):
        draw_path_on_floor(fl, path, ax, source=source,
                            target=target, bathroom_node=bathroom_node)

    color_legend = [
        mpatches.Patch(color='#2ecc71', label=f'Start ({source})'),
        mpatches.Patch(color='#3498db', label=f'End ({target})'),
        mpatches.Patch(color='#f39c12', label='Path node'),
        mpatches.Patch(color='crimson', label='Shortest path'),
    ]
    if bathroom_node is not None:
        color_legend.append(
            mpatches.Patch(color='#ff69b4', label=f'Bathroom ({bathroom_node})'))
    axes[0].legend(handles=color_legend, loc='upper left',
                   fontsize=8, title='Path', framealpha=0.85)

    fig.suptitle(
        f'Shortest path: {source} ({G.nodes[source]["floor"]}) → '
        f'{target} ({G.nodes[target]["floor"]})    '
        f'total cost = {total_cost:.2f}',
        fontsize=13)
    plt.tight_layout()
    plt.show()

    # text breakdown
    print('\nEdge-by-edge breakdown:')
    w = make_weight_fn(accessibility=False)
    running = 0.0
    for u, v in zip(path, path[1:]):
        d  = G[u][v]
        c  = w(u, v, d) or 0
        running += c
        marker = ''
        if 'transition' in d['edge_type']:
            marker = '   ⇅ FLOOR CHANGE'
        print(f"  {u:>3} → {v:<3}  type={d['edge_type']:<22s} "
              f"dist={d['distance']:>5}ft  cost={c:6.2f}{marker}")
    print(f'\nTotal cost: {total_cost:.2f}')
    print(f'Floors visited: {" → ".join(floors_in_path)}')


## 7. Interactive UI

Step through: **start floor → start node → end floor → end node →
bathroom / accessibility → Find Path**. The base map for the selected
floor refreshes whenever you change a dropdown so you can see where you
are.

In [8]:
def nodes_on_floor(fl):
    return sorted(n for n, a in G.nodes(data=True) if a.get('floor') == fl)


# ── Controls ──────────────────────────────────────────────────────────
start_floor = widgets.Dropdown(
    options=FLOOR_ORDER, value='ground',
    description='Start floor:',
    style={'description_width': 'initial'},
)
start_node = widgets.Dropdown(
    options=nodes_on_floor('ground'),
    description='Start node:',
    style={'description_width': 'initial'},
)
end_floor = widgets.Dropdown(
    options=FLOOR_ORDER, value='ground',
    description='End floor:',
    style={'description_width': 'initial'},
)
end_node = widgets.Dropdown(
    options=nodes_on_floor('ground'),
    description='End node:',
    style={'description_width': 'initial'},
)
bathroom_chk = widgets.Checkbox(value=False, description='Stop at bathroom')
access_chk   = widgets.Checkbox(value=False,
                                description='Need accessible path (no stairs)')
run_btn = widgets.Button(description='Find Shortest Path',
                          button_style='success', icon='map-marker')

start_map_out = widgets.Output()
end_map_out   = widgets.Output()
result_out    = widgets.Output()


def show_start_map(*_):
    with start_map_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        draw_floor_map(start_floor.value, ax=ax,
                        highlight=start_node.value, title_suffix=' — START')
        plt.tight_layout()
        plt.show()


def show_end_map(*_):
    with end_map_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        draw_floor_map(end_floor.value, ax=ax,
                        highlight=end_node.value, title_suffix=' — END')
        plt.tight_layout()
        plt.show()


def on_start_floor_change(_):
    opts = nodes_on_floor(start_floor.value)
    start_node.options = opts
    start_node.value = opts[0]
    show_start_map()


def on_end_floor_change(_):
    opts = nodes_on_floor(end_floor.value)
    end_node.options = opts
    end_node.value = opts[0]
    show_end_map()


def on_run(_):
    with result_out:
        clear_output(wait=True)
        s, t = start_node.value, end_node.value
        try:
            path, cost, bnode = shortest_path_with_options(
                G, s, t,
                accessibility=access_chk.value,
                bathroom=bathroom_chk.value,
            )
        except nx.NetworkXNoPath:
            print(f'No path between {s} and {t} with the current options.')
            if access_chk.value:
                print('  (Accessibility is on — try turning it off, or pick '
                      'nodes reachable via elevators only.)')
            return
        except nx.NodeNotFound as e:
            print(f'Node not found: {e}')
            return
        visualize_path(path, s, t, cost, bathroom_node=bnode)


start_floor.observe(on_start_floor_change, names='value')
end_floor  .observe(on_end_floor_change,   names='value')
start_node .observe(show_start_map,        names='value')
end_node   .observe(show_end_map,          names='value')
run_btn    .on_click(on_run)

# initial render
show_start_map()
show_end_map()

display(widgets.VBox([
    widgets.HTML('<h3>Step 1 — choose your starting floor and node</h3>'),
    widgets.HBox([start_floor, start_node]),
    start_map_out,
    widgets.HTML('<h3>Step 2 — choose your destination floor and node</h3>'),
    widgets.HBox([end_floor, end_node]),
    end_map_out,
    widgets.HTML('<h3>Step 3 — options</h3>'),
    widgets.HBox([bathroom_chk, access_chk]),
    run_btn,
    widgets.HTML('<h3>Result</h3>'),
    result_out,
]))
